# 5G deployment on local infrastructure: Demo DIGITAFRICA

#### **Objective**:

The goal of this notebook is to deploy a 5G network on an existing, local infrastructure, with real radio hardware, using the Post-5G orchestrator.

#### *What is the Post-5G orchestrator?*

It's a Python library that simplifies 5G deployments through a generic *metamodel* describing the 5G network parameters, and a *deployment model* describing the parametes and state of the 5G deployment and the underlying infrastructure.

It relies internally on [Kubernetes](https://kubernetes.io) to create a [cluster](https://kubernetes.io/docs/concepts/architecture/) where the 5G network containers will be deployed, and [Ansible](https://docs.ansible.com/projects/ansible/latest/getting_started/introduction.html) to control the infrastructure through [SSH](https://www.cloudflare.com/learning/access-management/what-is-ssh/) and automate the deployment tasks.

#### **0.**   Pre-requisites:

#### *Installing Dependencies*:

Before starting the 5G deployment, we will first install and prepare the Post-5G orchestartor dependencies.

Install the Post-5G orchestrator library:

In [1]:
! pip install --upgrade "git+https://gitlab.inria.fr/slices-ri/blueprints/post-5g/orchestrator.git" --extra-index-url=https://gitlab.inria.fr/api/v4/projects/70914/packages/pypi/simple

Looking in indexes: https://pypi.org/simple, https://gitlab.inria.fr/api/v4/projects/70914/packages/pypi/simple
  Cloning https://gitlab.inria.fr/slices-ri/blueprints/post-5g/orchestrator.git to /private/var/folders/4j/8j63qg_x4y93h7cg9lzd1n90k524rf/T/pip-req-build-w33vjckj
  Running command git clone --filter=blob:none --quiet https://gitlab.inria.fr/slices-ri/blueprints/post-5g/orchestrator.git /private/var/folders/4j/8j63qg_x4y93h7cg9lzd1n90k524rf/T/pip-req-build-w33vjckj
  Resolved https://gitlab.inria.fr/slices-ri/blueprints/post-5g/orchestrator.git to commit 9505f87eb2316b108af680c72ba870d7112c8376
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done

[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


Install the [Ansible](https://docs.ansible.com/projects/ansible/latest/getting_started/introduction.html) dependencies:

In [2]:
! ansible-galaxy install -r requirements.yaml

Starting galaxy collection install process
Nothing to do. All requested collections are already installed. If you want to reinstall them, consider using `--force`.


#### *5G deployment description*:

For this example, we will use:
- one Linux machine to deploy the 5G core using the open-source [OpenAirInterface](https://openairinterface.org) implementation,
- same machine will also be used to deploy the 5G RAN (Radio Access Network) using the open-source [OpenAirInterface](https://openairinterface.org) implementation as well
- [USRP B210](https://www.ettus.com/all-products/ub210-kit/) (`usrp01`) as Radio Unit (RU) for our 5G RAN
- Samsung S21 FE (`qdong01`) and a [Raspberry Pi 4](https://www.raspberrypi.com/products/raspberry-pi-4-model-b/) with a [Quectel RM520N-GL](https://www.quectel.com/product/5g-rm520n-series/) 5G modem (`qdong02`) as User Equipements

Now we will input the entire 5G network description to the Post-5G orchestrator using the *metamodel* and the *deployment model*.

**Metamodel:**

Using the Post-5G *metamodel*, we can describe our 5G network and all its parameters.

Since we are using real radio frequencies, we obtained the authorization for a 40 Mhz frequency range in NR band n78, from 3610 MHz to 3650 MHz.

> **Side Note**: In the 5G 3GPP standards, we describe radio frequencies using ARFCN: Absolute Radio-Frequency Channel Number. It is a unique code or identification number used in cellular networks to specify a specific radio frequency.
>
> (Reference: [Online ARFCN calculator](https://5g-tools.com/5g-nr-arfcn-calculator/))

For the first 20-Mhz half of the 40-Mhz frequency range, from 3610 MHz to 3630 MHz, use the following radio configuration for the gNB:

```bash
radioConfiguration:
    nrBand: 78
    downlinkArfcn: 641333 # central frequency ≈ 3620 MHz
    pointAFrequencyArfcn: 640722 # ≈ 3610.835 MHz
    channelBandwidthMhz: 20
    subcarrierSpacingKhz: 30
    ssbArfcn: 641376 # ≈ 3620.64 MHz
    ssbSubcarrierSpacing: 30
```

For the second half, use the following radio configuration parameters for the gNB:

```bash
radioConfiguration:
    nrBand: 78
    downlinkArfcn: 642667 # central frequency ≈ 3640.005 MHz
    pointAFrequencyArfcn: 642055 # ≈ 3630.825 MHz
    channelBandwidthMhz: 20
    subcarrierSpacingKhz: 30
    ssbArfcn: 642709 # ≈ 3640.635 MHz
    ssbSubcarrierSpacing: 30
```

For this deployment example, we are going to test with the first half of the authorized frequency range:

In [134]:
from metamodel.models.metamodel import MetaModel

metamodel = MetaModel.from_yaml(
"""
core:
  components:
    nrf:
      type: nrf
      config:
        plmnList:
          - plmnId:
              mcc: '001'
              mnc: '01'
            snssaiList:
              - internet

    ausf:
      type: ausf
      config:
        plmnList:
          - mcc: '001'
            mnc: '01'

    amf:
      type: amf
      config:
        guamiList:
          - plmnId:
              mcc: '001'
              mnc: '01'
        taiList:
          - plmnId:
              mcc: '001'
              mnc: '01'
            tac: 1
        plmnList:
          - plmnId:
              mcc: '001'
              mnc: '01'
            snssaiList:
              - internet

    smf:
      type: smf
      config:
        snssaiList:
          - internet
        plmnList:
          - mcc: '001'
            mnc: '01'
        upfs:
          - upf

    upf:
      type: upf
      config:
        snssaiList:
          - internet

    udr:
      type: udr

    udm:
      type: udm

ran:
  components:
    gnb:
      type: gnb
      config:
        gnbId: 1
        tac: 1
        plmnList:
          - plmnId:
              mcc: '001'
              mnc: '01'
            snssaiList:
              - internet
        nrCellId: 1
        physicalCellId: 0
        radioConfiguration:
          nrBand: 78
          downlinkArfcn: 641333 # central frequency ≈ 3620 MHz
          pointAFrequencyArfcn: 640722 # ≈ 3610.835 MHz
          channelBandwidthMhz: 20
          subcarrierSpacingKhz: 30
          ssbArfcn: 641376 # ≈ 3620.64 MHz
          ssbSubcarrierSpacing: 30
          tddConfiguration:
            periodicityMilliseconds: 5.0
            nrofDownlinkSlots: 7
            nrofDownlinkSymbols: 6
            nrofUplinkSlots: 2
            nrofUplinkSymbols: 4
        ru: usrp01
        amf: amf

rus:
  usrp01:
    type: usrp
    model: b210
    txAttenuationDB: 10
    rxAttenuationDB: 10
    txGainDB: 35
    rxGainDB: 114

ues:
  qdong01:
    type: cots
    config:
      supi: "001010000000017"
      pei: "868371054189101"
      authenticationData:
        k: "fec86ba6eb707ed08905757b1bb44b8f"
        opc: "C42449363BBAD02B66D16BC975D77CC1"
      subscriptions:
        - snssai: internet
          dnn: internet
      gnbs:
        - gnb
      nrBands:
        - 78

  qdong02:
    type: cots
    config:
      supi: "001010000000018"
      pei: "868371054166950"
      authenticationData:
        k: "fec86ba6eb707ed08905757b1bb44b8f"
        opc: "C42449363BBAD02B66D16BC975D77CC1"
      subscriptions:
        - snssai: internet
          dnn: internet
      gnbs:
        - gnb
      nrBands:
        - 78

slices:
  internet:
    sst: 1
    sd: 0xFFFFFF
    dnns:
      internet:
        ipv4Network: 11.1.1.0/24
"""
)

**Deployment model:**

Here, we describe the underlying infrastructure where the 5G network will be deployed:

In [136]:
from orchestrator.models.deployment import DeploymentModel

deployment_model = DeploymentModel.from_yaml(
"""
params:
  k8s:
    master_node: node01
    install_multus: false

nodes:
  node01:
    is_provisioned: true
    ssh_info:
      hostname: "192.168.1.156"
      user: ubuntu
      sudo_password: ubuntu
    roles:
      - core
      - ran

rus:
  usrp01:
    is_provisioned: true

core:
  implementation: oai

ran:
  implementation: oai
  components:
    gnb:
      type: gnb
      advanced_config:
        initialDLBWPlocationAndBandwidth: 13750
        initialDLBWPcontrolResourceSetZero: 12
        initialULBWPlocationAndBandwidth: 13750
        ssb_perRACH_OccasionAndCB_PreamblesPerSSB: 14
        pusch_TargetSNRx10: 250
        pucch_TargetSNRx10: 200
        prach_dtx_threshold: 120
        pucch0_dtx_threshold: 100
"""
)

Reference configuration: https://gitlab.eurecom.fr/oai/openairinterface5g/-/blob/develop/targets/PROJECTS/GENERIC-NR-5GC/CONF/gnb.sa.band78.fr1.106PRB.usrpb210.conf

> **Note**: To configure SSH on Ubuntu, you can follow [this](https://www.ionos.com/digitalguide/server/configuration/ubuntu-ssh/).

## **1.**   Kubernetes Deployement:

In [107]:
from orchestrator.deployment import K8SManagement

k8s = K8SManagement(metamodel=metamodel, deployment_model=deployment_model)

Deploy a Kubernetes cluster:

In [ ]:
await k8s.deploy()


PLAY [node01] ******************************************************************

TASK [k8s/create : Ensure Kubernetes directory exists on 'node01'] *************
[WARNING]: Host 'node01' is using the discovered Python interpreter at '/usr/bin/python3.12', but future installation of another Python interpreter could cause a different interpreter to be discovered. See https://docs.ansible.com/ansible-core/2.21/reference_appendices/interpreter_discovery.html for more information.
ok: [node01]

TASK [k8s/create : Write kubeadm configuration on 'node01'] ********************
ok: [node01]

TASK [k8s/create : Initialize the Kubernetes control plane on 'node01'] ********
ok: [node01]

TASK [k8s/create : Ensure kube config directory exists on 'node01' root] *******
changed: [node01]

TASK [k8s/create : Copy admin.conf to kube config on 'node01'] *****************
changed: [node01]

TASK [k8s/create : Install Calico operator CRDs on 'node01'] *******************
[WARNING]: kubernetes<24.2.0 is 

## **2.**   5G Core Deployement:

In [ ]:
from orchestrator.deployment import CoreManagement

core = CoreManagement(k8s)

In [138]:
await core.deploy()


PLAY [node01] ******************************************************************

PLAY [node01] ******************************************************************

TASK [helm/prepare : Create Helm build directory for 'oai-nrf' chart] **********

TASK [helm/prepare : Create Helm build directory for 'oai-database' chart] *****

PLAY [node01] ******************************************************************

TASK [helm/prepare : Create Helm build directory for 'oai-ausf' chart] *********

PLAY [node01] ******************************************************************

PLAY [node01] ******************************************************************

TASK [helm/prepare : Create Helm build directory for 'oai-upf' chart] **********

TASK [helm/prepare : Create Helm build directory for 'oai-udm' chart] **********

PLAY [node01] ******************************************************************

PLAY [node01] ******************************************************************

TASK [helm/prep

Check deployed [pods](https://kubernetes.io/docs/concepts/workloads/pods/) with `kubectl`:

In [139]:
from orchestrator.deployment import ProvisioningManagement

prov = ProvisioningManagement(core)

print((await prov.run_node_command(
    command=["kubectl", "get", "pods", "-n", "oai-core"],
    nodes=["node01"],
    become=True, quiet=True))["node01"])

NAME                            READY   STATUS    RESTARTS   AGE
oai-amf-7b4668cdb7-59hkl        1/1     Running   0          44s
oai-ausf-6c87fcdd5-zj7vl        1/1     Running   0          35s
oai-database-684b6f6fb5-mpz5q   1/1     Running   0          70s
oai-nrf-658cf56668-w5g7j        1/1     Running   0          50s
oai-smf-7c84f54549-dwf74        1/1     Running   0          29s
oai-udm-7595967696-sqrd6        1/1     Running   0          50s
oai-udr-7cd7fbf54f-ftj2c        1/1     Running   0          44s
oai-upf-5b76674866-v8n76        1/1     Running   0          21s


## **3.**   RAN deployement:

Check if USRP B210 is plugged in using `lsusb` command:

In [140]:
print((await prov.run_node_command(
    command=["lsusb"],
    nodes=["node01"],
    quiet=True))["node01"])

Bus 001 Device 001: ID 1d6b:0002 Linux Foundation 2.0 root hub
Bus 001 Device 002: ID 0bda:568c Realtek Semiconductor Corp. Integrated Webcam HD
Bus 002 Device 001: ID 1d6b:0003 Linux Foundation 3.0 root hub
Bus 002 Device 002: ID 2500:0020 Ettus Research LLC USRP B210


Tune the node by optimizing the CPU performance to ensure a more stable RAN deployment.

(Reference: https://kb.ettus.com/USRP_Host_Performance_Tuning_Tips_and_Tricks)

In [64]:
await prov.run_node_command(
    command=["apt", "install", "-y", "cpufrequtils"],
    nodes=["node01"], become=True, quiet=True)
await prov.run_node_command(
    command=["bash", "-c", "for ((i=0;i<$(nproc --all);i++)); do cpufreq-set -c $i -r -g performance; done"],
    nodes=["node01"], become=True, quiet=True)
await prov.run_node_command(
    command=["cpufreq-info"],
    nodes=["node01"], become=True, quiet=True)

{'node01': 'cpufrequtils 008: cpufreq-info (C) Dominik Brodowski 2004-2009\nReport errors and bugs to cpufreq@vger.kernel.org, please.\nanalyzing CPU 0:\n  driver: intel_pstate\n  CPUs which run at the same hardware frequency: 0\n  CPUs which need to have their frequency coordinated by software: 0\n  maximum transition latency: 4294.55 ms.\n  hardware limits: 400 MHz - 3.90 GHz\n  available cpufreq governors: performance, powersave\n  current policy: frequency should be within 400 MHz and 3.90 GHz.\n                  The governor "performance" may decide which speed to use\n                  within this range.\n  current CPU frequency is 3.90 GHz.\nanalyzing CPU 1:\n  driver: intel_pstate\n  CPUs which run at the same hardware frequency: 1\n  CPUs which need to have their frequency coordinated by software: 1\n  maximum transition latency: 4294.55 ms.\n  hardware limits: 400 MHz - 3.90 GHz\n  available cpufreq governors: performance, powersave\n  current policy: frequency should be with

Deploy OAI gNB:

In [141]:
from orchestrator.deployment import RANManagement

ran = RANManagement(core)

In [142]:
await ran.deploy()


PLAY [node01] ******************************************************************

TASK [helm/prepare : Create Helm build directory for 'oai-gnb' chart] **********
[WARNING]: Host 'node01' is using the discovered Python interpreter at '/usr/bin/python3.12', but future installation of another Python interpreter could cause a different interpreter to be discovered. See https://docs.ansible.com/ansible-core/2.21/reference_appendices/interpreter_discovery.html for more information.
ok: [node01]

TASK [helm/prepare : Copy local 'oai-gnb' Helm chart directory for 'oai-gnb'] ***
ok: [node01]

TASK [helm/prepare : Render 'oai-gnb' Helm Chart.yaml file from Jinja template] ***
ok: [node01]

TASK [helm/prepare : Render templated Helm files for 'oai-gnb'] ****************
changed: [node01] => (item=values.yaml)
changed: [node01] => (item=config.yaml)

PLAY RECAP *********************************************************************
node01                     : ok=4    changed=1    unreachable=0   

Check OAI gNB logs:

In [ ]:
print(await ran.get_logs(components=["gnb"], quiet=True))

/proc/sys/kernel/core_pattern=|/usr/share/apport/apport -p%p -s%s -c%c -d%d -P%P -u%u -g%g -F%F -- %E
No configuration file /opt/oai-gnb/etc/gnb.conf found: attempting to find YAML config
== Configuration file:

Active_gNBs:
  - gnb
# Asn1_verbosity, choice in: none, info, annoying
Asn1_verbosity: none
gNBs:
  ##### Identification parameters:
  - gNB_ID: 0x1

    gNB_name: gnb

    tracking_area_code: 1

    plmn_list:
      - mcc: 001
        mnc: 01
        mnc_length: 2
        snssaiList:
          - sst: 1
            sd: 0xFFFFFF

    nr_cellid: 1

    ##### Physical parameters:
    min_rxtxtime: 6

    servingCellConfigCommon:
      #spCellConfigCommon
      - physCellId: 0
        #  downlinkConfigCommon
        #frequencyInfoDL
        # this is 3300.60 MHz + 53*12*30e-3 MHz = 3319.68
        absoluteFrequencySSB: 641376
        # this is 3300.60 MHz
        dl_absoluteFrequencyPointA: 640722
        #scs-SpecificCarrierList
        dl_offstToCarrier: 0
        # subcarrierSpa

Check OAI AMF logs:

In [ ]:
print(await core.get_logs(components=["amf"], quiet=True))

Trying to read .yaml configuration file
LTTNG Tracing disabled at build-time!
[2026-10-13 15:02:15.820] [amf_app] [start] Options parsed!
[2026-10-13 15:02:15.820] [system ] [debug] Parsing the configuration file, file type YAML.
[2026-10-13 15:02:15.820] [config ] [info] Reading NF configuration from /openair-amf/etc/config.yaml
[2026-10-13 15:02:15.827] [config ] [debug] Validating configuration of log_level
[2026-10-13 15:02:15.831] [config] [info] ==== OAI-CN5G AMF vBranch: HEAD Abrev. Hash: 5ad2873c Date: Tue Dec 9 12:09:26 2025 +0000 ====
[2026-10-13 15:02:15.831] [config] [info] ======================    AMF   =====================
[2026-10-13 15:02:15.831] [config] [info] Configuration AMF:
[2026-10-13 15:02:15.831] [config] [info] - Instance ................: 1
[2026-10-13 15:02:15.831] [config] [info] - PID dir .................: /var/run
[2026-10-13 15:02:15.831] [config] [info] - AMF NAME.................: oai-amf
[2026-10-13 15:02:15.831] [config] [info] - GUAMI (MCC, MNC,

#### Cleanup:

To stop the gNB:

In [ ]:
# await ran.remove()


PLAY [node01] ******************************************************************

TASK [helm/uninstall : Uninstall 'oai-gnb' Helm chart] *************************
[WARNING]: Host 'node01' is using the discovered Python interpreter at '/usr/bin/python3.12', but future installation of another Python interpreter could cause a different interpreter to be discovered. See https://docs.ansible.com/ansible-core/2.21/reference_appendices/interpreter_discovery.html for more information.
[WARNING]: Deprecation warnings can be disabled by setting `deprecation_warnings=False` in ansible.cfg.
[DEPRECATION WARNING]: The 'status.values' return value is deprecated and will be removed in version 8.0.0. Use 'status.release_values' instead. This feature will be removed from collection 'kubernetes.core' version 8.0.0.
changed: [node01]

TASK [helm/uninstall : Wait for 'oai-gnb' pod to be deleted] *******************
FAILED - RETRYING: [node01]: helm/uninstall : Wait for 'oai-gnb' pod to be deleted (59 ret

To stop the 5G core:

In [ ]:
# await core.remove()

To do a full teardown of the Kubernetes deployment:

In [ ]:
# await k8s.remove()